In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
print(os.listdir("/content/drive/MyDrive"))

['SB Collect.PDF', '2304.py', 'Classroom', 'Sudents Paper Registration Form.xlsx', 'IN819853_1 (1).jpg', 'IN819853_1.jpg', 'exam fee receipt sem 2.pdf', '2304 Set A train reservation.py', 'SB Collect.pdf', '2304 ADHIT AMONKAR REAPEAT EXAMINATION.py', 'Ml backpro', 'SmartSelect_20250404_220701_One UI Home (1).jpg', 'SmartSelect_20250404_220701_One UI Home.jpg', '20250404_222223 (1).jpg', '20250404_222223.jpg', 'SB Collect semester 4.pdf', 'Ml assingment PCA.pdf', '2304_output.py', '2304_outptut (1).pdf', '2304_outptut.pdf', 'Adhit_2304.mp3', 'SEA-Batch-A-2304.gdoc', 'SEA-Batch-A-2304.pdf', 'Copy of SEA-Batch-A-2304.gdoc', 'Adhit (2).pdf', 'Adhit (1).pdf', 'Adhit.pdf', 'Assignment 2.1.pdf', 'Os_assignment.pdf', 'linux os.docx', 'Adhit new Aadhaar.pdf', 'prject1.docx', 'SB Collect SEMESTER 5 FEE.pdf', 'Lab Assignment 01 (1).gdoc', 'Lab Assignment 01.gdoc', 'Lab Assignment 02.gdoc', 'Research Area-.gdoc', 'Research Area-.docx', 'Importing csv to numpy.gdoc', 'full.ipynb', 'output.txt', 'Sh

In [3]:
data_dir = "/content/drive/MyDrive/dataset_svarah"

In [4]:
files = os.listdir(data_dir)

wav_count = len([f for f in files if f.endswith(".wav")])
txt_count = len([f for f in files if f.endswith(".txt")])

print("WAV:", wav_count)
print("TXT:", txt_count)

WAV: 6656
TXT: 6656


In [5]:
!cp -r "/content/drive/MyDrive/dataset_svarah" /content/

In [6]:
data_dir = "/content/dataset_svarah"

In [7]:
!pip install transformers==4.41.2 datasets torchaudio jiwer

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 77.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 64.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.7.1
    Uninstalling huggingface_hub-1.7.1:
      Successfully uninstalled huggingface_hub-1.7.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [8]:
!pip uninstall -y peft

Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1


In [10]:
import zipfile
import os

zip_path = "/content/final_model.zip"
extract_path = "/content/final_model"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extracted model ")

Extracted model 


In [11]:
import os
import re
from datasets import Dataset, Audio

data = []

for file in os.listdir(data_dir):
    if file.endswith(".wav"):
        base = file[:-4]
        wav_path = os.path.join(data_dir, file)
        txt_path = os.path.join(data_dir, base + ".txt")

        if os.path.exists(txt_path):
            with open(txt_path, "r", encoding="utf-8") as f:
                text = f.read().lower()

                # CLEAN TEXT (CRITICAL)
                text = re.sub(r"[^a-zA-Z\s]", "", text)
                text = re.sub(r"\s+", " ", text).strip()

            if len(text) == 0:
                continue

            data.append({
                "audio": wav_path,
                "text": text
            })

dataset = Dataset.from_list(data)
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

dataset = dataset.train_test_split(test_size=0.1)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['audio', 'text'],
        num_rows: 5913
    })
    test: Dataset({
        features: ['audio', 'text'],
        num_rows: 657
    })
})


In [12]:
import json

all_text = " ".join([item["text"] for item in data])

vocab = list(set(all_text))

vocab_dict = {v: k for k, v in enumerate(sorted(vocab))}
vocab_dict["|"] = vocab_dict[" "]
del vocab_dict[" "]

vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

with open("vocab.json", "w") as f:
    json.dump(vocab_dict, f)

print(" Vocab created")

 Vocab created


In [13]:
from transformers import Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2Processor

tokenizer = Wav2Vec2CTCTokenizer(
    "vocab.json",
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|"
)

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=False
)

processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer
)

In [14]:
def prepare(batch):
    audio = batch["audio"]

    batch["input_values"] = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_values[0]

    batch["labels"] = processor.tokenizer(batch["text"]).input_ids

    return batch

dataset = dataset.map(
    prepare,
    remove_columns=dataset["train"].column_names
)

Map:   0%|          | 0/5913 [00:00<?, ? examples/s]

Map:   0%|          | 0/657 [00:00<?, ? examples/s]

In [ ]:
sample = dataset["train"][0]
print(sample["labels"][:20])

[1, 12, 5, 24, 1, 9, 0, 14, 5, 5, 4, 0, 1, 0, 19, 9, 24, 0, 1, 13]


In [15]:
from transformers import Wav2Vec2ForCTC

model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/wav2vec2-base-960h",
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
)

model.freeze_feature_encoder()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Some weights of the model checkpoint at facebook/wav2vec2-base-960h were not used when initializing Wav2Vec2ForCTC: ['wav2vec2.encoder.pos_conv_embed.conv.weight_g', 'wav2vec2.encoder.pos_conv_embed.conv.weight_v']
- This IS expected if you are initializing Wav2Vec2ForCTC from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing Wav2Vec2ForCTC from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original1', 'wav2vec2.masked_spec_embed']
You sho

In [16]:
from dataclasses import dataclass
from typing import Dict, List, Union
import torch

@dataclass
class DataCollatorCTCWithPadding:
    processor: any
    padding: Union[bool, str] = True

    def __call__(self, features):
        input_features = [{"input_values": f["input_values"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.pad(input_features, return_tensors="pt")

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor)

In [17]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./wav2vec2-finetuned",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    learning_rate=3e-5,
    logging_steps=100,
    save_steps=400,
    fp16=True,
)

In [18]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
)

In [ ]:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Step,Training Loss
100,46.320900
200,7.753400
300,4.301900
400,5.266700
500,4.798100
600,4.137800
700,3.949100
800,5.404500
900,3.818500
1000,3.647200


TrainOutput(global_step=4434, training_loss=4.6146440174754915, metrics={'train_runtime': 2292.4642, 'train_samples_per_second': 7.738, 'train_steps_per_second': 1.934, 'total_flos': 1.2157055163000893e+18, 'train_loss': 4.6146440174754915, 'epoch': 2.9989854582346975})

In [ ]:
from transformers import Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2Processor

# tokenizer from your vocab
tokenizer = Wav2Vec2CTCTokenizer(
    "/content/vocab.json",
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|"
)

# feature extractor
feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=False
)

# processor
processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer
)

In [ ]:
from transformers import Wav2Vec2ForCTC

model = Wav2Vec2ForCTC.from_pretrained(
    "/content/wav2vec2-finetuned/checkpoint-4400"
)

In [ ]:
processor.save_pretrained("/content/final_model")
model.save_pretrained("/content/final_model")

In [ ]:
import librosa

audio_path = "/content/PTT-20260326-WA0000.wav"

speech, sr = librosa.load(audio_path, sr=16000)

In [ ]:
inputs = processor(
    speech,
    sampling_rate=16000,
    return_tensors="pt",
    padding=True
)

In [ ]:
import torch

with torch.no_grad():
    logits = model(inputs.input_values).logits

pred_ids = torch.argmax(logits, dim=-1)

In [ ]:
!zip -r final_model.zip /content/final_model

  adding: content/final_model/ (stored 0%)
  adding: content/final_model/config.json (deflated 65%)
  adding: content/final_model/added_tokens.json (deflated 17%)
  adding: content/final_model/special_tokens_map.json (deflated 41%)
  adding: content/final_model/model.safetensors (deflated 9%)
  adding: content/final_model/tokenizer_config.json (deflated 69%)
  adding: content/final_model/preprocessor_config.json (deflated 38%)
  adding: content/final_model/vocab.json (deflated 56%)


In [19]:
#  Load your saved model correctly

from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

model = Wav2Vec2ForCTC.from_pretrained("/content/final_model/content/final_model")
processor = Wav2Vec2Processor.from_pretrained("/content/final_model/content/final_model")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [ ]:
# Continue training for 4 more epochs

from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./wav2vec2-finetuned",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    num_train_epochs=4,   # ONLY extra epochs
    learning_rate=3e-5,
    logging_steps=100,
    save_steps=400,
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
)

trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Step,Training Loss
100,1.740700
200,1.804400
300,2.520900
400,1.473500
500,1.554900
600,1.287700
700,1.389400
800,1.475300
900,1.311900
1000,1.179100


TrainOutput(global_step=5912, training_loss=1.1361244578161485, metrics={'train_runtime': 2961.4592, 'train_samples_per_second': 7.987, 'train_steps_per_second': 1.996, 'total_flos': 1.6374570911168517e+18, 'train_loss': 1.1361244578161485, 'epoch': 3.9986472776462634})

In [ ]:
!pip install jiwer

In [ ]:
#  LOADING MODEL

from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import torch

model = Wav2Vec2ForCTC.from_pretrained("/content/final_model5600/final_model_5600")
processor = Wav2Vec2Processor.from_pretrained("/content/final_model5600/final_model_5600")

model.eval()

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Wav2Vec2ForCTC(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2GroupNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
          (activation): GELUActivation()
          (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
        )
        (1-4): 4 x Wav2Vec2NoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2NoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projection): Linear(in_features=512, out_features=768, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder)

In [20]:
#  METRICS SETUP

from jiwer import wer, cer
import time

def evaluate_sample(audio, sr, reference_text):
    start_time = time.time()

    inputs = processor(audio, sampling_rate=sr, return_tensors="pt")

    with torch.no_grad():
        logits = model(**inputs).logits

    pred_ids = torch.argmax(logits, dim=-1)
    prediction = processor.batch_decode(pred_ids)[0]

    end_time = time.time()

    #  Metrics
    wer_score = wer(reference_text, prediction)
    cer_score = cer(reference_text, prediction)

    latency = end_time - start_time
    audio_duration = len(audio) / sr
    rtfx = latency / audio_duration   # Real-Time Factor

    return {
        "prediction": prediction,
        "wer": wer_score,
        "cer": cer_score,
        "latency_sec": latency,
        "rtfx": rtfx
    }


In [22]:
import torch
import time
from jiwer import wer, cer, process_words

model.eval()

results = []

#  Helper functions
def compute_ser(reference, prediction):
    return 0 if reference.strip() == prediction.strip() else 1

def compute_word_metrics(reference, prediction):
    output = process_words(reference, prediction)

    TP = output.hits
    FP = output.insertions + output.substitutions
    FN = output.deletions + output.substitutions

    return TP, FP, FN

def compute_accuracy_precision(reference, prediction):
    TP, FP, FN = compute_word_metrics(reference, prediction)

    total = TP + FP + FN
    accuracy = TP / total if total > 0 else 0
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0

    return accuracy, precision


#  Main loop
for sample in dataset["test"]:
    input_values = torch.tensor(sample["input_values"]).unsqueeze(0)

    start_time = time.time()

    with torch.no_grad():
        logits = model(input_values).logits

    pred_ids = torch.argmax(logits, dim=-1)
    prediction = processor.batch_decode(pred_ids)[0]

    end_time = time.time()

    # Decode reference
    label_ids = sample["labels"]
    label_ids = [id for id in label_ids if id != -100]
    reference = processor.tokenizer.decode(label_ids)

    #  Existing metrics
    wer_score = wer(reference, prediction)
    cer_score = cer(reference, prediction)

    latency = end_time - start_time
    audio_duration = len(sample["input_values"]) / 16000
    rtfx = latency / audio_duration

    #  New metrics
    ser_score = compute_ser(reference, prediction)
    accuracy, precision = compute_accuracy_precision(reference, prediction)

    results.append({
        "wer": wer_score,
        "cer": cer_score,
        "latency": latency,
        "rtfx": rtfx,
        "ser": ser_score,
        "accuracy": accuracy,
        "precision": precision
    })


#  Averages
avg_wer = sum(r["wer"] for r in results) / len(results)
avg_cer = sum(r["cer"] for r in results) / len(results)
avg_latency = sum(r["latency"] for r in results) / len(results)
avg_rtfx = sum(r["rtfx"] for r in results) / len(results)

avg_ser = sum(r["ser"] for r in results) / len(results)
avg_accuracy = sum(r["accuracy"] for r in results) / len(results)
avg_precision = sum(r["precision"] for r in results) / len(results)


#  Output
print("WER:", avg_wer)
print("CER:", avg_cer)
print("SER:", avg_ser)
print("Accuracy:", avg_accuracy)
print("Precision:", avg_precision)
print("Latency:", avg_latency)
print("RTFx:", avg_rtfx)

WER: 0.4920845332335959
CER: 0.1931877575500974
SER: 0.9299847792998478
Accuracy: 0.43379227775088275
Precision: 0.5491559845548474
Latency: 1.4021530267491915
RTFx: 0.2753852223001529


In [ ]:
#  Use your SAME processor setup + load checkpoint model

from transformers import Wav2Vec2ForCTC, Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2Processor

#  Recreate processor (same as before)
tokenizer = Wav2Vec2CTCTokenizer(
    "/content/vocab.json",
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|"
)

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=False
)

processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer
)

#  Load checkpoint model
model = Wav2Vec2ForCTC.from_pretrained("/content/wav2vec2-finetuned/checkpoint-5600")

#  Save BOTH properly
save_path = "/content/final_model2"

model.save_pretrained(save_path)
processor.save_pretrained(save_path)

print(" Saved complete model at:", save_path)

In [ ]:
#  Zip the folder

!zip -r /content/final_model_5600.zip /content/final_model_5600

  adding: content/final_model_5600/ (stored 0%)
  adding: content/final_model_5600/preprocessor_config.json (deflated 38%)
  adding: content/final_model_5600/added_tokens.json (deflated 17%)
  adding: content/final_model_5600/tokenizer_config.json (deflated 69%)
  adding: content/final_model_5600/special_tokens_map.json (deflated 41%)
  adding: content/final_model_5600/config.json (deflated 65%)
  adding: content/final_model_5600/vocab.json (deflated 56%)
  adding: content/final_model_5600/model.safetensors (deflated 7%)
